# 02 — Data Collection: Path B (Concurrent Client Classes)

**Issue:** [#24 — M2 Round 1B](https://github.com/nholguinrh/DAMO-699-Capstone-project-GRP5/issues/24)  
**Owner:** Lerneir · **Reviewer:** Nelson  
**Approach:** Per-source client classes (one per API), fetched concurrently via `ThreadPoolExecutor`, config-driven series list.

This notebook runs the full data-collection pipeline end-to-end:
1. Imports the orchestrator and configuration from `src/`
2. Fetches all series from BoC Valet, FRED, and StatCan **concurrently**
3. Caches raw responses as timestamped JSON in `data/raw/`
4. Displays a summary of what was pulled

**Pre-requisite:** Set the FRED API key environment variable before running:  
```
# PowerShell
$env:FRED_API_KEY = "your-key-here"

# Bash / Zsh
export FRED_API_KEY="your-key-here"
```

> **Scope:** This notebook intentionally stops at raw-data caching (Bronze layer).  
> Silver/Gold layer transformations are handled by later pipeline stages.

In [ ]:
# ── Imports & setup ──────────────────────────────────────────────────────
import sys
import logging
from pathlib import Path

# Ensure the project root is on the Python path so `src` is importable
PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Configure logging so we can see what the clients are doing
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)-8s  %(message)s",
    datefmt="%H:%M:%S",
)

import src.config as cfg
from src.orchestrator import run_full_collection

print(f"Date range : {cfg.DATE_START} → {cfg.DATE_END}")
print(f"Cache dir  : {cfg.RAW_DATA_DIR}")
print(f"BoC series : {len(cfg.BOC_SERIES)} + USD/CAD stitched")
print(f"FRED series: {len(cfg.FRED_SERIES)}")
print(f"StatCan    : {len(cfg.STATCAN_VECTORS)} vector(s)")

In [ ]:
# ── Run the concurrent data-collection pipeline ───────────────────────
results = run_full_collection(cfg)

In [ ]:
# ── Summary: what was pulled ─────────────────────────────────────────
print("\n" + "=" * 64)
print("DATA COLLECTION SUMMARY")
print("=" * 64)

for source, series_data in sorted(results.items()):
    print(f"\n📦 {source.upper()}")
    for name, data in series_data.items():
        if isinstance(data, dict):
            obs = data.get("observations", [])
            count = len(obs) if isinstance(obs, list) else "N/A"
        elif isinstance(data, list):
            # StatCan WDS returns a list of objects
            count = sum(
                len(item.get("object", {}).get("vectorDataPoint", []))
                for item in data
                if isinstance(item, dict)
            )
        else:
            count = "unknown"
        print(f"  • {name}: {count} observations")

print("\n" + "=" * 64)

In [ ]:
# ── Verify: list all cached JSON files ───────────────────────────────
cached_files = sorted(cfg.RAW_DATA_DIR.glob("*.json"))
print(f"\n📁 Cached files in {cfg.RAW_DATA_DIR}:")
print(f"   Total: {len(cached_files)} file(s)\n")

for f in cached_files:
    size_kb = f.stat().st_size / 1024
    print(f"  {f.name:.<60s} {size_kb:>7.1f} KB")

---

## ✅ Definition of Done (Issue #24)

- [x] Pulls all series listed in `04_data_sources.md` §4.4 end-to-end without manual intervention
- [x] Raw pulls cached as timestamped local JSON, reproducible offline
- [x] No hardcoded credentials — FRED API key read from environment variable

**Next step:** This raw data feeds into the Silver/Gold layer pipeline (Issue #30 — Feature Engineering).